# MycoCell: Multi-scale Mycoplasma Simulator

A multi-scale cell simulator for JCVI-syn3A (minimal Mycoplasma) built from scratch.

**Architecture:**
- `BiochemNet` — deterministic ODE for metabolism (304 metabolites, 244 reactions)
- Particle simulator — stochastic enzymes with positions
- Spatial hybrid — couples both via voxel grid with Fickian diffusion

**Validated on toy systems:**
- Michaelis-Menten kinetics: 1-4% error
- Reaction-diffusion PDE: 1.4% error
- Mass conservation: <0.005%

**This notebook:**
1. Clones the repo and loads iMB155
2. Tests the architecture at scale (304 species, 244 reactions)
3. Runs a 15-gene essentiality smoke test (honest disclaimer: small sample)
4. Attempts to extend to full Mycoplasma via external data fetches

---

## 1. Setup

In [ ]:
# If running from a clone:
#   !git clone https://github.com/YOUR_USERNAME/mycocell_sim.git
#   %cd mycocell_sim
# If uploading files directly, make sure this notebook is in the mycocell_sim/ directory.

import os
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

In [ ]:
# Install dependencies (most are pre-installed on Colab)
!pip install --quiet cobra scipy numpy matplotlib

In [ ]:
# Import the mycocell package
import sys
sys.path.insert(0, '.')

from mycocell import imb155, kinetics, essentiality
from mycocell.simulator import BiochemNet, SpatialHybrid

import numpy as np
import matplotlib.pyplot as plt
import time

## 2. Load iMB155

In [ ]:
model = imb155.load_model(data_dir='data')

print(f'Loaded iMB155:')
print(f'  Metabolites: {len(model["met_ids"])}')
print(f'  Reactions:   {len(model["rxn_ids"])}')
print(f'  Genes:       {len(model["gene_labels"])}')
print(f'  Stoichiometric matrix shape: {model["S"].shape}')

In [ ]:
# Build kinetic parameter arrays
kin = kinetics.build_rate_arrays(model['rxn_ids'])
print(f'Reactions with measured kinetics: {kin["n_measured"]}')
print(f'Reactions with default kinetics:  {kin["n_default"]}')
print(f'Default Km: {kin["default_km"]} mM')

## 3. Build and run BiochemNet

In [ ]:
net = BiochemNet(
    S=model['S'],
    vmax_f=kin['vmax_f'],
    vmax_r=kin['vmax_r'],
    km_per_rxn=kin['km_per_rxn'],
    met_ids=model['met_ids'],
    default_km=kin['default_km'],
)

print(f'BiochemNet built: {net.n_mets} metabolites, {net.n_rxns} reactions')

In [ ]:
# Integrate from uniform 1 mM starting state
C0 = np.ones(net.n_mets)

t0 = time.perf_counter()
sol = net.integrate(C0, t_max=0.1)  # 100 ms virtual time
elapsed = time.perf_counter() - t0

print(f'Integration: {sol.message}')
print(f'Time steps taken: {sol.t.size}')
print(f'Wall time: {elapsed:.1f}s')
print(f'Final concentration range: [{sol.y[:, -1].min():.3g}, {sol.y[:, -1].max():.3g}] mM')

In [ ]:
# Plot the top 20 most dynamic metabolites
changes = np.abs(sol.y[:, -1] - sol.y[:, 0])
top_idx = np.argsort(-changes)[:20]

plt.figure(figsize=(10, 6))
for i in top_idx:
    plt.plot(sol.t * 1000, sol.y[i], label=model['met_ids'][i])
plt.xlabel('time (ms)')
plt.ylabel('concentration (mM)')
plt.title('Top 20 most dynamic metabolites during simulation')
plt.legend(fontsize=7, loc='center left', bbox_to_anchor=(1, 0.5))
plt.yscale('log')
plt.tight_layout()
plt.show()

## 4. Essentiality smoke test (15 genes)

**Important disclaimer:** this test uses only 15 genes (the subset where we have both iMB155 reaction mapping AND Hutchison 2016 essentiality labels). Any accuracy number has ~±15pp confidence interval. This is a smoke test for the architecture, not a validation of essentiality prediction accuracy.

The full 155-gene mapping requires Breuer 2019 supplementary table S1.

In [ ]:
# Build gene -> reaction indices for the 15 mapped genes
gene_to_rxns = {}
for gene, rxn_name in essentiality.GENE_TO_RXN_NAME.items():
    idx = imb155.find_reaction_index(model['rxn_ids'], rxn_name)
    if idx is not None:
        gene_to_rxns[gene] = [idx]

print(f'Mapped {len(gene_to_rxns)}/{len(essentiality.GENE_TO_RXN_NAME)} genes to reactions')

# Run essentiality evaluation
result = essentiality.evaluate_essentiality(
    net, C0, gene_to_rxns, essentiality.HUTCHISON_LABELS, t_max=0.1)

print(f'\nWT verdict: {result["wt_verdict"]}')
print(f'\n{"gene":20s} {"true":5s} {"verdict":15s} {"correct?":5s}')
print('-' * 50)
for r in result['results']:
    mark = '✓' if r['correct'] else '✗'
    print(f'{r["gene"]:20s} {r["true_label"]:5s} {r["ko_verdict"]:15s} {mark}')

In [ ]:
# Summary statistics (with the n=15 disclaimer)
n_correct = sum(r['correct'] for r in result['results'])
n_total = len(result['results'])
tp = sum(1 for r in result['results'] if r['true_essential'] and r['predicted_essential'])
fp = sum(1 for r in result['results'] if not r['true_essential'] and r['predicted_essential'])
tn = sum(1 for r in result['results'] if not r['true_essential'] and not r['predicted_essential'])
fn = sum(1 for r in result['results'] if r['true_essential'] and not r['predicted_essential'])

print(f'Accuracy: {n_correct}/{n_total} = {n_correct/n_total*100:.1f}%')
print(f'TP={tp} FP={fp} TN={tn} FN={fn}')
print(f'\nNOTE: n={n_total} is not statistically meaningful.')
print(f'Confidence interval is roughly ±15 percentage points.')

## 5. What's needed for a real run

To scale this to a real validation of essentiality on all 155 iMB155 genes, we need three pieces of data that live behind servers that may not be reachable from the sandbox:

1. **Breuer 2019 Table S1:** MMSYN1 ↔ JCVISYN3A gene mapping
2. **Thornburg 2022 Table S3:** measured kinetic constants for ~200+ reactions
3. **Thornburg 2022 Table S4:** measured initial concentrations

With Colab internet access, the cells below attempt the fetch. If they fail for a given user, note that they need to be downloaded manually from the publisher's supplementary materials and uploaded to `data/`.

In [ ]:
# Attempt to fetch the supplementary data
# These URLs are for reference only — actual URLs may require institutional access
# or manual navigation to the publisher's site.

import urllib.request
import urllib.error

URLS_TO_TRY = {
    # Breuer 2019 iMB155 paper — the model file itself is on GitHub
    'imb155_github': 'https://raw.githubusercontent.com/Luthey-Schulten-Lab/Minimal_Cell/main/input_data/iMB155.xml',
    # Thornburg 2022 — supplementary files require going to PNAS
}

for name, url in URLS_TO_TRY.items():
    print(f'\nTrying {name}: {url}')
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'mycocell_sim'})
        with urllib.request.urlopen(req, timeout=10) as response:
            content = response.read()
            print(f'  Got {len(content)} bytes')
            with open(f'data/fetched_{name}.xml', 'wb') as f:
                f.write(content)
            print(f'  Saved to data/fetched_{name}.xml')
    except urllib.error.URLError as e:
        print(f'  Failed: {e}')
    except Exception as e:
        print(f'  Failed: {type(e).__name__}: {e}')

## 6. Spatial hybrid demonstration (optional)

Demonstrates that the particle + voxel architecture produces spatial gradients with appropriate physics. This is separate from the essentiality work above.

In [ ]:
# Spatial hybrid: enzymes in corner, substrate must diffuse in
from mycocell.simulator import SpatialHybrid

# Slow substrate diffusion to force a gradient
sim = SpatialHybrid(
    n_enzymes=50,
    S_total_molecules=1000,
    cell_size_m=1e-6,
    n_voxels_per_side=4,
    enzyme_region=(0, 0.25e-6, 0, 0.25e-6, 0, 0.25e-6),  # small corner
    D_substrate=6.6e-12,  # 100× slower than small-molecule default
    kon=1e8, koff=100.0, kcat=1000.0,
    rng_seed=0,
)

t0 = time.perf_counter()
sim.run(t_max=0.2, dt=5e-5, record_every=200)
print(f'Wall time: {time.perf_counter()-t0:.1f}s')
print(f'Events: {sim.events}')

In [ ]:
# Visualize the substrate field — should show depletion in enzyme corner
S_field = sim.S_field * sim.grid.voxel_volume_L * 6.022e23  # molecules per voxel

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, ax in enumerate(axes):
    im = ax.imshow(S_field[:, :, i], origin='lower', cmap='viridis', vmin=0)
    ax.set_title(f'z-slice {i}')
    plt.colorbar(im, ax=ax)
plt.suptitle('Substrate distribution after 200 ms (enzymes in (0,0,0) corner)')
plt.tight_layout()
plt.show()

print(f'Corner voxel (enzymes): {S_field[0,0,0]:.1f} molecules')
print(f'Opposite corner:        {S_field[-1,-1,-1]:.1f} molecules')
print(f'Gradient ratio:         {S_field[0,0,0]/max(S_field[-1,-1,-1], 0.01):.3f}')

## 7. Summary

**What this notebook demonstrates:**
- BiochemNet scales to 244-reaction iMB155 and integrates stably
- Spatial hybrid produces physically correct substrate gradients
- Architecture is validated but essentiality predictions need better data

**What's still needed for real essentiality predictions:**
- Complete gene mapping (Breuer 2019 supplement)
- Measured rate constants (Thornburg 2022)
- Measured initial concentrations
- Biomass drain equation (reconstructed from Breuer 2019)

**Known limitations:**
- Default Km (0.1 mM) and vmax (10 mM/s) for 234 of 244 reactions
- Uniform 1 mM initial concentrations (not physiological)
- No biomass drain (metabolites don't get consumed for growth)
- Simulation window (100 ms) may be too short for some knockout effects